# Show3D

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/docs/tutorials/show3d.ipynb)

`Show3D` scrubs a 3D stack slice by slice: a focal series, time series, tomographic reconstruction, or a sequence of related images. Drag the slider, press the play controls, or use the arrow keys to move through the stack.

This tutorial uses a real gold HAADF image from [`bobleesj/quantem-data`](https://huggingface.co/datasets/bobleesj/quantem-data). The built-in tutorial loader makes a calibrated moving-crop stack from that image, so the documentation and Colab examples use real microscope data while still loading quickly.

```{tip}
Run this exact notebook with the Colab badge above, or [View or download this notebook on GitHub](https://github.com/bobleesj/quantem.widget/blob/main/docs/tutorials/show3d.ipynb). For finished results, use [Saving and sharing](widget_export) to export interactive HTML or share a trusted notebook with widget state.
```


In [1]:
import subprocess
import sys

import numpy as np
import torch

try:
    import google.colab  # noqa: F401
except Exception:
    pass
else:
    from google.colab import output

    output.enable_custom_widget_manager()
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/bobleesj/quantem.widget.git"],
        check=True,
    )

from quantem.widget import Show3D
from quantem.widget.data import load_tutorial_show3d

volume_dataset = load_tutorial_show3d(n_frames=32, stride=8, crop_size=256)


Source: gold_haadf_npy from Hugging Face
Full image: 4096 x 4096 uint16
Stack: (32, 256, 256), stride 8, pixel size 0.1489 nm


## Scrub the real-data stack

The helper returns a quantem `Dataset3d`, so depth and lateral calibration travel with the stack. `Show3D` reads that metadata automatically and draws a physical scale bar without widget-level pixel-size arguments.

In [2]:
Show3D(volume_dataset)

Show3D(32×256×256, frame=16, cmap=plasma)

## Page through related 3D views

Pages are useful when every parameter value has the same multi-panel movie layout. This synthetic torch example keeps the notebook light and fast while exercising the same paged Show3D controls used for real reconstruction sweeps.


In [3]:
torch.manual_seed(4)
page_labels = ["lambda 0.01", "lambda 0.03", "lambda 0.10"]
panel_titles = ["raw", "filtered", "residual", "probe"]

frames, n = 16, 144
t = torch.linspace(0, 1, frames)[:, None, None]
y, x = torch.meshgrid(
    torch.linspace(-1, 1, n),
    torch.linspace(-1, 1, n),
    indexing="ij",
)
x = x[None]
y = y[None]
center_x = -0.45 + 0.90 * t
center_y = 0.18 * torch.sin(2 * torch.pi * t)
base_stack = torch.exp(-((x - center_x) ** 2 + (y - center_y) ** 2) / 0.060)
base_stack += 0.45 * torch.exp(-((x + 0.28) ** 2 + (y + 0.35) ** 2) / 0.035)
base_stack += 0.12 * torch.sin(18 * x + 6 * t) * torch.cos(14 * y)
base_stack = base_stack.to(torch.float32)

page_stacks = []
for lam, denoise, artifact_strength in zip([0.01, 0.03, 0.10], [0.15, 0.42, 0.70], [0.20, 0.10, 0.03]):
    noise = artifact_strength * torch.randn_like(base_stack)
    drift = artifact_strength * torch.sin((16 + 80 * lam) * x + 4 * t) * torch.cos(12 * y)
    raw = base_stack + noise + drift
    filtered = (1 - denoise) * raw + denoise * base_stack.mean(dim=0, keepdim=True)
    residual = raw - filtered
    probe = torch.roll(base_stack, shifts=int(90 * lam), dims=2) + 0.4 * lam * torch.cos(24 * x) * torch.cos(24 * y)
    page_stacks.append(torch.stack([raw, filtered, residual, probe]))
page_stacks = torch.stack(page_stacks).numpy().astype(np.float32)


In [4]:
paged_show3d = Show3D(
    page_stacks,
    panel_titles=panel_titles,
    page_labels=page_labels,
    max_cols=4,
    sampling=0.03,
    units="nm",
    show_fft=False,
    link_contrast=False,
)
paged_show3d.star_page(1)
paged_show3d


Show3D(16×144×1728, frame=8, cmap=plasma)

## Trigger a fresh render in an existing widget

For live stacks, display one `Show3D` object and update it with `set_image()` as new frames arrive. `set_image()` is the render trigger: it writes a fresh current-frame transfer, bumps the frame sequence used by the frontend renderer, invalidates playback buffers, and clamps the current slice to the new stack.

Use `offline=False` for live-growing stacks. Small examples can otherwise choose the offline notebook path, which is useful for saved notebooks but is not the path you want when a loop keeps appending frames.


In [5]:
live_frames = [frame for frame in volume_dataset.array[:4]]
live3d = Show3D(
    np.stack(live_frames),
    labels=[f"frame {i + 1}" for i in range(len(live_frames))],
    offline=False,
    fps=4,
)
live3d


Show3D(4×256×256, frame=2, cmap=plasma)

In [6]:
live_frames.extend(volume_dataset.array[4:8])
live3d.set_image(
    np.stack(live_frames),
    labels=[f"frame {i + 1}" for i in range(len(live_frames))],
)
live3d.slice_idx = len(live_frames) - 1
